In [ ]:
#All the imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.stats import norm
from scipy.stats import kurtosis
from scipy.ndimage import maximum_filter, label
import multiprocessing as mp
import os
from time import perf_counter

# shared code and paths (see README.md and shared/)
import sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "paths.py").exists())
sys.path.insert(0, str(REPO))
from shared.fields import partial_x_fft, partial_y_fft
from shared.filtering import gaussian_fft_filter_2d
from shared.loading import grid_spacing, load, output_times
from shared.paths import BRMS, results_dir, run_dir
from shared.plotting import save_figure, use_style
use_style()
RESULTS = results_dir("quality_checks", "Jrms")

In [ ]:
# ---- runs, grid and settings ----
RUNS = [run_dir(beta_p=1, beta_e=1, brms=BRMS)]
DX, DY = grid_spacing(RUNS[0])
FILTER_WIDTH_FRACTION = 0.2          # Gaussian filter width k0, as a fraction of the number of cells
N_WORKERS = os.cpu_count()

In [ ]:
def J_par_rms_at(job):
    """(time, rms of J_par after the Gaussian filter) at one output time of one run."""
    run, output_time = job
    Bx, By = load("Bx", output_time, run), load("By", output_time, run)

    # Filtering
    filter_width = FILTER_WIDTH_FRACTION * len(Bx)
    Bx_filtered = gaussian_fft_filter_2d(Bx, k0=filter_width)
    By_filtered = gaussian_fft_filter_2d(By, k0=filter_width)

    J_par = partial_x_fft(By_filtered, DX) - partial_y_fft(Bx_filtered, DY)
    return output_time, np.sqrt(np.mean(J_par**2))


def J_par_rms_series(run):
    """J_par rms (filtered) at every output time of the run, computed in parallel."""
    jobs = [(run, output_time) for output_time in output_times(run)]
    start = perf_counter()
    with mp.Pool(processes=N_WORKERS) as pool:
        results = sorted(pool.map(J_par_rms_at, jobs))
    print(f"Total runtime: {perf_counter() - start:.2f} seconds")
    return np.array([output_time for output_time, _ in results]), np.array([rms for _, rms in results])


if __name__ == "__main__":
    J_rms_filtered = {run: J_par_rms_series(run) for run in RUNS}     # run -> (times, J_rms)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# No filter
# for sim_dir, J_rms in all_J_rms_nofilter.items():
#     label = sim_dir.split('/')[-1] + " (no filter)"
#     plt.plot(timesteps, J_rms, label=label)

# Filtered
for run, (times, J_rms) in J_rms_filtered.items():
    ax.plot(times, J_rms, '--', label=f"{run.parent.name}/{run.name} (filtered)")  # dashed for filtered

ax.set_xlabel('Time')
ax.set_ylabel(r'$J_{rms}$')
ax.set_title('J_rms vs Time for Different Simulations')
ax.legend()
ax.grid(True)
save_figure(fig, RESULTS, "J_rms_vs_time")
plt.show()

In [ ]:
np.max(J_rms)

In [ ]:
PEAK_FRACTION = 0.95                 # times where J_rms is within 5% of its maximum
J_max = np.max(J_rms)
indices_near_peak = np.where(J_rms >= PEAK_FRACTION * J_max)[0]
times_near_peak = times[indices_near_peak]
values_near_peak = J_rms[indices_near_peak]

In [ ]:
print(indices_near_peak)